# Evaluation Notebook — Engineer 5

End-to-end evaluation of 6 best-fold supervised checkpoints
on the MVTec AD test sets.

For each of the 6 categories (bottle, capsule, carpet, hazelnut,
leather, pill), this notebook:

1. Loads the best-fold checkpoint (selected by val AUROC)
2. Runs inference on the test set
3. Computes AUROC, AUPR, F1, Accuracy with 95% bootstrap CIs
4. Saves ROC, PR, and confusion matrix plots
5. Generates a Grad-CAM gallery and a worst-20 failure-case gallery
6. Aggregates everything into `results/eval/summary.json`

The output of this notebook is the empirical evidence for the
final report (`docs/REPORT.md`).

**Inputs**
- `MVTec AD` dataset (Kaggle): 6 categories, ~600 test images per category
- `ssidds-checkpoints` dataset (Kaggle): 18 supervised checkpoints
- `results/best_checkpoints.json` from GitHub: which fold to use per category

**Compute**: GPU P100 — runtime ≈ 20-30 minutes

In [1]:
# ──────────────────────────────────────────────────────────────────────
# Parameters — edit these before running
# ──────────────────────────────────────────────────────────────────────

# GitHub repo to pull the latest evaluation code from
GITHUB_REPO   = "https://github.com/ManuelGamal/Self-Supervised-Industrial-Defect-Detection-System"
GITHUB_BRANCH = "main"

# Kaggle paths (auto-mounted by Kaggle for the attached datasets)
MVTEC_INPUT_DIR    = "/kaggle/input/datasets/ipythonx/mvtec-ad"
CHECKPOINTS_INPUT  = "/kaggle/input/datasets/manuelgamal/ssidds-checkpoints/checkpoints/supervised"

# Working / output directories
WORKDIR = "/kaggle/working/ssidds"           # cloned repo lives here
RESULTS = "/kaggle/working/eval_results"     # all evaluation artefacts go here

# Bootstrap configuration
N_BOOTSTRAP = 10_000   # 10k resamples for 95% CIs
SEED        = 42

# Galleries
N_GRADCAM_PER_CLASS = 5    # 5 normal + 5 anomalous per category
N_WORST_FAILURES    = 20   # worst 20 by |score - label|

# Per-checkpoint inference settings
DEVICE      = "cuda"   # P100 GPU
BATCH_SIZE  = 32
NUM_WORKERS = 2

# ──────────────────────────────────────────────────────────────────────
# Fold-naming convention bridge
# ──────────────────────────────────────────────────────────────────────
# results.parquet stores folds as 1-indexed (1, 2, 3).
# Kaggle dataset folders are 0-indexed (fold_0, fold_1, fold_2).
def fold_dirname(fold_from_parquet: int) -> str:
    """Convert 1-indexed parquet fold number to Kaggle folder name."""
    return f"fold_{fold_from_parquet - 1}"


print("Configuration loaded:")
print(f"  MVTEC_INPUT_DIR    = {MVTEC_INPUT_DIR}")
print(f"  CHECKPOINTS_INPUT  = {CHECKPOINTS_INPUT}")
print(f"  WORKDIR            = {WORKDIR}")
print(f"  RESULTS            = {RESULTS}")
print(f"  DEVICE             = {DEVICE}")
print(f"  N_BOOTSTRAP        = {N_BOOTSTRAP}")
print(f"  fold_dirname(1)    = {fold_dirname(1)}")
print(f"  fold_dirname(2)    = {fold_dirname(2)}")
print(f"  fold_dirname(3)    = {fold_dirname(3)}")

Configuration loaded:
  MVTEC_INPUT_DIR    = /kaggle/input/datasets/ipythonx/mvtec-ad
  CHECKPOINTS_INPUT  = /kaggle/input/datasets/manuelgamal/ssidds-checkpoints/checkpoints/supervised
  WORKDIR            = /kaggle/working/ssidds
  RESULTS            = /kaggle/working/eval_results
  DEVICE             = cuda
  N_BOOTSTRAP        = 10000
  fold_dirname(1)    = fold_0
  fold_dirname(2)    = fold_1
  fold_dirname(3)    = fold_2


In [2]:
# ──────────────────────────────────────────────────────────────────────
# Sanity check — verify both Kaggle datasets are mounted at the
# expected paths before we install anything or do real work
# ──────────────────────────────────────────────────────────────────────
import os

EXPECTED_CATEGORIES = ["bottle", "capsule", "carpet", "hazelnut", "leather", "pill"]


def _check(label, path, must_contain=None):
    print(f"\n{label}: {path}")
    print(f"  exists? {os.path.isdir(path)}")
    if not os.path.isdir(path):
        return False
    contents = sorted(os.listdir(path))
    print(f"  entries: {contents[:10]}{' ...' if len(contents) > 10 else ''}")
    if must_contain:
        missing = [m for m in must_contain if m not in contents]
        if missing:
            print(f"  ❌ MISSING: {missing}")
            return False
        print(f"  ✅ all required categories present")
    return True


print("=" * 60)
print("DATASET PATHS SANITY CHECK")
print("=" * 60)

ok_mvtec = _check("MVTEC_INPUT_DIR", MVTEC_INPUT_DIR, must_contain=EXPECTED_CATEGORIES)
ok_ckpts = _check("CHECKPOINTS_INPUT", CHECKPOINTS_INPUT, must_contain=EXPECTED_CATEGORIES)

print("\n" + "=" * 60)
if not (ok_mvtec and ok_ckpts):
    raise RuntimeError(
        "One or more required dataset paths are wrong or incomplete. "
        "Check the dataset slugs and re-attach if needed before continuing."
    )
print("✅ All dataset paths look good. Safe to proceed.")
print("=" * 60)

DATASET PATHS SANITY CHECK

MVTEC_INPUT_DIR: /kaggle/input/datasets/ipythonx/mvtec-ad
  exists? True
  entries: ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'license.txt', 'metal_nut', 'pill'] ...
  ✅ all required categories present

CHECKPOINTS_INPUT: /kaggle/input/datasets/manuelgamal/ssidds-checkpoints/checkpoints/supervised
  exists? True
  entries: ['bottle', 'capsule', 'carpet', 'hazelnut', 'leather', 'pill']
  ✅ all required categories present

✅ All dataset paths look good. Safe to proceed.


In [3]:
# ──────────────────────────────────────────────────────────────────────
# Setup — clone repo, install only what's strictly needed for evaluation
#
# CRITICAL: every pip install uses --no-deps so it doesn't downgrade
# Kaggle's preinstalled numpy 2.x. Kaggle's scipy/sklearn are compiled
# against numpy 2.x and crash with `numpy.dtype size changed` if we
# let pip swap in numpy 1.26.
# ──────────────────────────────────────────────────────────────────────

import os
import shutil
import subprocess
import sys
from pathlib import Path


def run(cmd, cwd=None, allow_fail=False):
    print(f"$ {' '.join(map(str, cmd))}")
    result = subprocess.run(cmd, cwd=cwd, check=False)
    if result.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")


def get_secret(name):
    from kaggle_secrets import UserSecretsClient
    return UserSecretsClient().get_secret(name)


# ─── 1. WANDB key (optional — only used if you want to log to W&B) ─────
try:
    os.environ["WANDB_API_KEY"] = get_secret("WANDB_API_KEY")
    print("[OK] WANDB_API_KEY loaded from Kaggle Secrets")
except Exception as exc:
    print(f"[WARN] WANDB key not loaded: {exc}")
    print("       This is fine — we don't need W&B for evaluation.")


# ─── 2. Clone the GitHub repo (fresh every time) ───────────────────────
if Path(WORKDIR).exists():
    shutil.rmtree(WORKDIR)
run([
    "git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH,
    GITHUB_REPO, WORKDIR,
])
print(f"[OK] Repo cloned into {WORKDIR}")


# ─── 3. Install ONLY what's needed for evaluation, with --no-deps ─────
# Skipped on purpose:
#   - albumentations  (only used for training augmentation)
#   - hydra-core      (only used by training entrypoint)
#   - pretty-errors   (cosmetic)
# These three pin numpy 1.x and would otherwise corrupt the env.
run([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    "pytorch-lightning==2.3.0",
    "torchmetrics==1.4.0",
    "timm==1.0.7",
    "grad-cam==1.5.0",
    "lightning-utilities",
    "ttach",
])
print("[OK] Evaluation dependencies installed (--no-deps)")


# ─── 4. Install our project as an editable package ────────────────────
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], cwd=WORKDIR)
print("[OK] Project installed (editable)")


# ─── 5. Make src importable from this notebook ────────────────────────
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)
print(f"[OK] {WORKDIR} added to sys.path")


# ─── 6. Verify numpy is on the right major version ────────────────────
import numpy as np
print(f"\n[CHECK] numpy version: {np.__version__}")
assert np.__version__.startswith("2."), (
    f"numpy is {np.__version__} — expected 2.x. "
    "Restart the kernel and re-run from the top. "
    "If this keeps happening, check that no cell pip-installs "
    "albumentations or anything that pins numpy<2."
)
print("[OK] numpy 2.x confirmed")


# ─── 7. Sanity-import all Engineer 5 modules ──────────────────────────
print("\n[INFO] Importing Engineer 5 modules...")
from src.evaluation.metrics import (
    compute_auroc, compute_aupr, compute_f1_optimal, compute_accuracy,
)
from src.evaluation.bootstrap import bootstrap_ci
from src.evaluation.evaluator import evaluate_checkpoint
from src.evaluation.qualitative import (
    gradcam_gallery, failure_case_gallery, run_qualitative_for_category,
)
from src.types import MetricsWithCI

print("[OK] All Engineer 5 modules imported successfully!")
print("\nSetup complete — ready to evaluate.")

[OK] WANDB_API_KEY loaded from Kaggle Secrets
$ git clone --depth 1 --branch main https://github.com/ManuelGamal/Self-Supervised-Industrial-Defect-Detection-System /kaggle/working/ssidds


Cloning into '/kaggle/working/ssidds'...


[OK] Repo cloned into /kaggle/working/ssidds
$ /usr/bin/python3 -m pip install -q --no-deps pytorch-lightning==2.3.0 torchmetrics==1.4.0 timm==1.0.7 grad-cam==1.5.0 lightning-utilities ttach
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 812.2/812.2 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.8/868.8 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 63.3 MB/s eta 0:00:00
[OK] Evaluation dependencies installed (--no-deps)
$ /usr/bin/python3 -m pip install -q --no-deps -e .
[OK] Project installed (editable)
[OK] /kaggle/working/ssidds added to sys.path

[CHECK] numpy version: 2.0.2
[OK] numpy 2.x confirmed

[INFO] Importing Engineer 5 modules...
[OK] All Engineer 5 modules imported successfully!

Setup complete — ready to evaluate.


In [4]:
# ──────────────────────────────────────────────────────────────────────
# Helper — build a test DataLoader for one MVTec category
#
# We use the project's MVTecDataset (CSV-driven) with split_filter="test".
# The split CSVs live in the cloned repo at data/splits/.
# ──────────────────────────────────────────────────────────────────────

from typing import List
import torch
from torch.utils.data import DataLoader

from src.data.mvtec_dataset import MVTecDataset
from src.types import Sample


def _sample_collate(batch: List[Sample]) -> Sample:
    """Collate a list of Sample dataclasses into one batched Sample."""
    return Sample(
        image=torch.stack([s.image for s in batch]),
        label=[s.label for s in batch],
        category=[s.category for s in batch],
        path=[s.path for s in batch],
    )


def build_test_dataloader(category: str) -> DataLoader:
    """Return a DataLoader over the test split for a single category."""
    split_csv = Path(WORKDIR) / "data" / "splits" / f"{category}_100pct_seed42.csv"
    if not split_csv.exists():
        raise FileNotFoundError(f"Split CSV not found: {split_csv}")

    dataset = MVTecDataset(
        root=MVTEC_INPUT_DIR,
        category=category,
        split_csv=split_csv,
        split_filter="test",
    )
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        collate_fn=_sample_collate,
    )


# Quick smoke-test on bottle so we know data loading works before we
# burn time on the full 6-category evaluation
print("Smoke-testing test dataloader on `bottle`...")
_test_dl = build_test_dataloader("bottle")
print(f"  bottle test set size: {len(_test_dl.dataset)} samples")
_first_batch = next(iter(_test_dl))
print(f"  first batch images:  {_first_batch.image.shape}")
print(f"  first batch labels:  {_first_batch.label[:8]} ...")
print("[OK] Dataloader works.")

Smoke-testing test dataloader on `bottle`...
  bottle test set size: 44 samples
  first batch images:  torch.Size([32, 3, 224, 224])
  first batch labels:  [0, 1, 0, 0, 0, 0, 1, 0] ...
[OK] Dataloader works.


In [5]:
# Quick peek at what's in the new _metadata block
import json
with open(Path(WORKDIR) / "results" / "best_checkpoints.json") as f:
    raw = json.load(f)
if "_metadata" in raw:
    print("Found _metadata block:")
    print(json.dumps(raw["_metadata"], indent=2))
else:
    print("No _metadata block in the JSON")

Found _metadata block:
{
  "source_parquet": "C:/Users/acer/Downloads/results (1).parquet",
  "selection_metric_used": "val_auroc",
  "tie_break_metric_used": "val_f1",
  "note": "Detailed run breakdown includes validation metrics and run URLs."
}


In [6]:
# ──────────────────────────────────────────────────────────────────────
# Load best_checkpoints.json (which fold to use per category)
# Schema is the one Engineer 4 produced — uses `selected_fold` and includes
# a `_metadata` block. We adapt to it here so the rest of the notebook is
# unchanged.
# ──────────────────────────────────────────────────────────────────────

import json

best_checkpoints_path = Path(WORKDIR) / "results" / "best_checkpoints.json"
if not best_checkpoints_path.exists():
    raise FileNotFoundError(
        f"{best_checkpoints_path} not found. Did the GitHub clone include "
        "results/best_checkpoints.json? If not, push it first."
    )

with open(best_checkpoints_path) as f:
    BEST_RAW = json.load(f)

# Engineer 4's schema:
#   { "_metadata": {...},
#     "bottle":  {"selected_fold": 3, "val_auroc": 0.99, "val_f1": 0.83, ...},
#     ... }
#
# We normalize each entry to a small dict our loop expects:
#   {"best_fold": int, "val_auroc": float, "val_aupr": float | None, "val_f1": float}
def _normalize_entry(entry: dict) -> dict | None:
    """Map Engineer 4's schema to our internal one. Return None if the
    entry doesn't look like a category record."""
    if not isinstance(entry, dict):
        return None
    fold = entry.get("selected_fold", entry.get("best_fold"))
    auroc = entry.get("val_auroc")
    if fold is None or auroc is None:
        return None
    return {
        "best_fold": int(fold),
        "val_auroc": float(auroc),
        # Engineer 4's JSON omits AUPR — fall back to F1 for the display
        # table; the real AUPR will be computed by the evaluator on the
        # actual test set anyway.
        "val_aupr": float(entry.get("val_aupr", entry.get("val_f1", 0.0))),
        "val_f1": float(entry.get("val_f1", 0.0)),
        # Carry through Engineer 4's enriched fields for traceability
        "wandb_run_id": entry.get("run_id"),
        "run_name": entry.get("run_name"),
    }


BEST = {}
skipped = []
for k, v in BEST_RAW.items():
    if k.startswith("_"):
        skipped.append(k)
        continue
    norm = _normalize_entry(v)
    if norm is None:
        skipped.append(k)
        continue
    BEST[k] = norm

if skipped:
    print(f"[INFO] Skipped non-category keys: {skipped}")
print(f"Loaded {best_checkpoints_path}")
print(f"  {len(BEST)} categories: {sorted(BEST.keys())}")
print()
print(f"  {'category':<10} {'fold':>5}  {'val_auroc':>10}  {'val_f1':>8}")
print(f"  {'-'*10} {'-'*5}  {'-'*10}  {'-'*8}")
for cat in sorted(BEST.keys()):
    info = BEST[cat]
    print(
        f"  {cat:<10} "
        f"{info['best_fold']:>5}  "
        f"{info['val_auroc']:>10.4f}  "
        f"{info['val_f1']:>8.4f}"
    )

# Build an explicit list of (category, parquet_fold, kaggle_ckpt_path) tuples
EVAL_TARGETS = []
for cat in sorted(BEST.keys()):
    info = BEST[cat]
    parquet_fold = info["best_fold"]
    kaggle_dir = fold_dirname(parquet_fold)
    ckpt_path = Path(CHECKPOINTS_INPUT) / cat / kaggle_dir / "best.ckpt"
    EVAL_TARGETS.append((cat, parquet_fold, ckpt_path))

print("\nResolved Kaggle checkpoint paths:")
for cat, fold, ckpt in EVAL_TARGETS:
    print(f"  {cat:<10} fold={fold} -> {ckpt}")
    if not ckpt.exists():
        raise FileNotFoundError(f"Checkpoint missing: {ckpt}")
print(f"\n[OK] All {len(EVAL_TARGETS)} checkpoints found on Kaggle.")

[INFO] Skipped non-category keys: ['_metadata']
Loaded /kaggle/working/ssidds/results/best_checkpoints.json
  6 categories: ['bottle', 'capsule', 'carpet', 'hazelnut', 'leather', 'pill']

  category    fold   val_auroc    val_f1
  ---------- -----  ----------  --------
  bottle         3      0.9939    0.8387
  capsule        1      0.9551    0.7451
  carpet         1      0.9664    0.8000
  hazelnut       1      0.9951    0.7500
  leather        3      1.0000    1.0000
  pill           3      0.9154    0.7647

Resolved Kaggle checkpoint paths:
  bottle     fold=3 -> /kaggle/input/datasets/manuelgamal/ssidds-checkpoints/checkpoints/supervised/bottle/fold_2/best.ckpt
  capsule    fold=1 -> /kaggle/input/datasets/manuelgamal/ssidds-checkpoints/checkpoints/supervised/capsule/fold_0/best.ckpt
  carpet     fold=1 -> /kaggle/input/datasets/manuelgamal/ssidds-checkpoints/checkpoints/supervised/carpet/fold_0/best.ckpt
  hazelnut   fold=1 -> /kaggle/input/datasets/manuelgamal/ssidds-checkpoints

In [7]:
# ──────────────────────────────────────────────────────────────────────
# Quantitative evaluation — for each best-fold checkpoint:
#   load → inference → metrics + bootstrap CIs → save plots
#
# This is the headline result. Expect ~2-3 minutes per category on P100.
# ──────────────────────────────────────────────────────────────────────

import time
from datetime import timedelta

# Make sure the output directory exists
Path(RESULTS).mkdir(parents=True, exist_ok=True)

all_metrics = []
total_start = time.time()

for cat, parquet_fold, ckpt_path in EVAL_TARGETS:
    print(f"\n{'='*60}")
    print(f"  Evaluating  {cat:<10}  (fold {parquet_fold})")
    print(f"{'='*60}")

    test_dl = build_test_dataloader(cat)
    print(f"  test samples: {len(test_dl.dataset)}")

    t0 = time.time()
    metrics = evaluate_checkpoint(
        checkpoint_path=ckpt_path,
        dataloader=test_dl,
        category=cat,
        fold=parquet_fold,
        device=DEVICE,
        output_dir=RESULTS,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED,
    )
    elapsed = time.time() - t0

    all_metrics.append(metrics)

    print(f"  AUROC    = {metrics.image_auroc:.4f}  "
          f"[95% CI: {metrics.image_auroc_ci[0]:.4f}, {metrics.image_auroc_ci[1]:.4f}]")
    print(f"  AUPR     = {metrics.image_aupr:.4f}  "
          f"[95% CI: {metrics.image_aupr_ci[0]:.4f}, {metrics.image_aupr_ci[1]:.4f}]")
    print(f"  F1       = {metrics.f1:.4f}  "
          f"[95% CI: {metrics.f1_ci[0]:.4f}, {metrics.f1_ci[1]:.4f}]  "
          f"@ thr={metrics.threshold:.4f}")
    print(f"  Accuracy = {metrics.accuracy:.4f}  "
          f"[95% CI: {metrics.accuracy_ci[0]:.4f}, {metrics.accuracy_ci[1]:.4f}]")
    print(f"  n_test   = {metrics.n_samples}")
    print(f"  elapsed  = {timedelta(seconds=int(elapsed))}")

    # Free GPU memory before the next category
    import torch as _torch
    _torch.cuda.empty_cache()

total_elapsed = time.time() - total_start
print(f"\n{'='*60}")
print(f"  Quantitative evaluation done in {timedelta(seconds=int(total_elapsed))}")
print(f"  Plots + metrics.json saved under {RESULTS}/<category>/fold<N>/")
print(f"{'='*60}")


  Evaluating  bottle      (fold 3)
  test samples: 44


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.6.1, which is newer than your current Lightning version: v2.3.0


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

  AUROC    = 0.9746  [95% CI: 0.9235, 1.0000]
  AUPR     = 0.9094  [95% CI: 0.7120, 1.0000]
  F1       = 0.8571  [95% CI: 0.6667, 1.0000]  @ thr=0.3255
  Accuracy = 0.9318  [95% CI: 0.8409, 1.0000]
  n_test   = 44
  elapsed  = 0:00:41

  Evaluating  capsule     (fold 1)
  test samples: 53


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.6.1, which is newer than your current Lightning version: v2.3.0


...

  AUROC    = 0.9846  [95% CI: 0.9531, 1.0000]
  AUPR     = 0.9330  [95% CI: 0.7925, 1.0000]
  F1       = 0.8696  [95% CI: 0.6667, 1.0000]  @ thr=0.2554
  Accuracy = 0.9605  [95% CI: 0.9079, 1.0000]
  n_test   = 76
  elapsed  = 0:00:39

  Evaluating  leather     (fold 3)
  test samples: 56


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.6.1, which is newer than your current Lightning version: v2.3.0


  AUROC    = 0.9966  [95% CI: 0.9837, 1.0000]
  AUPR     = 0.9911  [95% CI: 0.9556, 1.0000]
  F1       = 0.9630  [95% CI: 0.8571, 1.0000]  @ thr=0.5226
  Accuracy = 0.9821  [95% CI: 0.9464, 1.0000]
  n_test   = 56
  elapsed  = 0:00:38

  Evaluating  pill        (fold 3)
  test samples: 66


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.6.1, which is newer than your current Lightning version: v2.3.0


  AUROC    = 0.9360  [95% CI: 0.8669, 0.9837]
  AUPR     = 0.9048  [95% CI: 0.7979, 0.9739]
  F1       = 0.8085  [95% CI: 0.6667, 0.9167]  @ thr=0.4983
  Accuracy = 0.8636  [95% CI: 0.7727, 0.9394]
  n_test   = 66
  elapsed  = 0:00:38

  Quantitative evaluation done in 0:03:54
  Plots + metrics.json saved under /kaggle/working/eval_results/<category>/fold<N>/


In [8]:
# ──────────────────────────────────────────────────────────────────────
# Qualitative galleries — Grad-CAM overlays + failure-case grids
#
# For each category, generate:
#   - gradcam_gallery.png     (5 normal + 5 anomalous, with attention overlays)
#   - failure_cases.png       (worst 20 by |score - label|)
#
# These figures feed the qualitative + negative-results sections of REPORT.md.
# ──────────────────────────────────────────────────────────────────────

import torch
from src.models.lit_module import DefectClassifier

galleries_dir = Path(RESULTS) / "galleries"
galleries_dir.mkdir(parents=True, exist_ok=True)

t0 = time.time()
for cat, parquet_fold, ckpt_path in EVAL_TARGETS:
    print(f"\n[Gallery] {cat} (fold {parquet_fold}) ...")

    # Reload the model — cheap (~1s) and avoids any state from the
    # quantitative loop leaking into the qualitative analysis.
    model = DefectClassifier.load_from_checkpoint(
        str(ckpt_path), map_location=DEVICE
    )
    test_dl = build_test_dataloader(cat)

    gradcam_path, failures_path = run_qualitative_for_category(
        model=model,
        dataloader=test_dl,
        category=cat,
        output_dir=galleries_dir,
        device=DEVICE,
        n_per_class=N_GRADCAM_PER_CLASS,
        n_worst=N_WORST_FAILURES,
    )
    print(f"  Grad-CAM: {gradcam_path}")
    print(f"  Failures: {failures_path}")

    del model
    torch.cuda.empty_cache()

elapsed = time.time() - t0
print(f"\nQualitative galleries done in {timedelta(seconds=int(elapsed))}")
print(f"Saved under: {galleries_dir}")


[Gallery] bottle (fold 3) ...


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.6.1, which is newer than your current Lightning version: v2.3.0


  Grad-CAM: /kaggle/working/eval_results/galleries/bottle/gradcam_gallery.png
  Failures: /kaggle/working/eval_results/galleries/bottle/failure_cases.png

[Gallery] capsule (fold 1) ...
  Grad-CAM: /kaggle/working/eval_results/galleries/capsule/gradcam_gallery.png
  Failures: /kaggle/working/eval_results/galleries/capsule/failure_cases.png

[Gallery] carpet (fold 1) ...
  Grad-CAM: /kaggle/working/eval_results/galleries/carpet/gradcam_gallery.png
  Failures: /kaggle/working/eval_results/galleries/carpet/failure_cases.png

[Gallery] hazelnut (fold 1) ...
  Grad-CAM: /kaggle/working/eval_results/galleries/hazelnut/gradcam_gallery.png
  Failures: /kaggle/working/eval_results/galleries/hazelnut/failure_cases.png

[Gallery] leather (fold 3) ...
  Grad-CAM: /kaggle/working/eval_results/galleries/leather/gradcam_gallery.png
  Failures: /kaggle/working/eval_results/galleries/leather/failure_cases.png

[Gallery] pill (fold 3) ...
  Grad-CAM: /kaggle/working/eval_results/galleries/pill/gradcam_g

In [9]:
# ──────────────────────────────────────────────────────────────────────
# Aggregate all metrics into one summary file for the report
#
# Writes:
#   - {RESULTS}/summary.json       (one entry per category, full precision)
#   - {RESULTS}/summary_table.md   (markdown table with 95% CIs, ready to
#                                   paste into docs/REPORT.md)
# ──────────────────────────────────────────────────────────────────────

# 1. JSON summary (machine-readable)
summary = {}
for m in all_metrics:
    summary[m.category] = {
        "fold":        m.fold,
        "n_samples":   m.n_samples,
        "threshold":   m.threshold,
        "image_auroc": m.image_auroc,
        "image_auroc_ci": list(m.image_auroc_ci),
        "image_aupr":  m.image_aupr,
        "image_aupr_ci":  list(m.image_aupr_ci),
        "f1":          m.f1,
        "f1_ci":       list(m.f1_ci),
        "accuracy":    m.accuracy,
        "accuracy_ci": list(m.accuracy_ci),
    }

summary_path = Path(RESULTS) / "summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"[OK] Wrote {summary_path}")


# 2. Markdown table (paste-ready for the report)
def _ci(v, lo, hi):
    return f"{v:.4f} [{lo:.4f}, {hi:.4f}]"

md_lines = [
    "| Category | Fold | n_test | Threshold | AUROC (95% CI) | AUPR (95% CI) | F1 (95% CI) | Accuracy (95% CI) |",
    "|----------|-----:|-------:|----------:|----------------|---------------|-------------|-------------------|",
]
for cat in sorted(summary.keys()):
    s = summary[cat]
    md_lines.append(
        f"| {cat} | {s['fold']} | {s['n_samples']} | {s['threshold']:.4f} "
        f"| {_ci(s['image_auroc'], *s['image_auroc_ci'])} "
        f"| {_ci(s['image_aupr'],  *s['image_aupr_ci'])} "
        f"| {_ci(s['f1'],          *s['f1_ci'])} "
        f"| {_ci(s['accuracy'],    *s['accuracy_ci'])} |"
    )

# Overall row (mean across categories — a useful summary, not a CI)
import statistics as _stats
md_lines.append(
    f"| **Overall (mean)** |  | {sum(s['n_samples'] for s in summary.values())} |  "
    f"| {_stats.mean(s['image_auroc'] for s in summary.values()):.4f} "
    f"| {_stats.mean(s['image_aupr']  for s in summary.values()):.4f} "
    f"| {_stats.mean(s['f1']           for s in summary.values()):.4f} "
    f"| {_stats.mean(s['accuracy']     for s in summary.values()):.4f} |"
)

table_path = Path(RESULTS) / "summary_table.md"
with open(table_path, "w") as f:
    f.write("\n".join(md_lines) + "\n")
print(f"[OK] Wrote {table_path}")


# 3. Print the table for immediate inspection
print("\n" + "=" * 80)
print("FINAL EVALUATION TABLE (paste-ready for docs/REPORT.md)")
print("=" * 80)
for line in md_lines:
    print(line)

[OK] Wrote /kaggle/working/eval_results/summary.json
[OK] Wrote /kaggle/working/eval_results/summary_table.md

FINAL EVALUATION TABLE (paste-ready for docs/REPORT.md)
| Category | Fold | n_test | Threshold | AUROC (95% CI) | AUPR (95% CI) | F1 (95% CI) | Accuracy (95% CI) |
|----------|-----:|-------:|----------:|----------------|---------------|-------------|-------------------|
| bottle | 3 | 44 | 0.3255 | 0.9746 [0.9235, 1.0000] | 0.9094 [0.7120, 1.0000] | 0.8571 [0.6667, 1.0000] | 0.9318 [0.8409, 1.0000] |
| capsule | 1 | 53 | 0.4617 | 0.8834 [0.7745, 0.9615] | 0.8093 [0.6200, 0.9384] | 0.7407 [0.5000, 0.9000] | 0.8679 [0.7736, 0.9434] |
| carpet | 1 | 60 | 0.3225 | 0.9574 [0.9021, 0.9926] | 0.8726 [0.6984, 0.9759] | 0.8125 [0.6316, 0.9412] | 0.9000 [0.8167, 0.9667] |
| hazelnut | 1 | 76 | 0.2554 | 0.9846 [0.9531, 1.0000] | 0.9330 [0.7925, 1.0000] | 0.8696 [0.6667, 1.0000] | 0.9605 [0.9079, 1.0000] |
| leather | 3 | 56 | 0.5226 | 0.9966 [0.9837, 1.0000] | 0.9911 [0.9556, 1.0000] | 